# 🚀 Fine-tune Qwen 4B (Friendly 'Macha' Model) with QLoRA
This notebook walks you through fine-tuning a small instruct model using QLoRA. We will use the dataset you generated!

### 1. Install Dependencies
We use Unsloth because it is significantly faster and uses less memory for QLoRA training.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 2. Load the Model (Qwen2.5-3B-Instruct or 4B)
We'll load the model in 4-bit quantization to save GPU memory.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

# We use Qwen 2.5 3B Instruct as our strong base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 4. Upload and Load Your Custom Dataset
Run this cell. It will prompt you to upload `dataset.jsonl` directly into the notebook. Once uploaded, it automatically loads it.

In [ ]:
from google.colab import files
from datasets import load_dataset
import os

# 1. Prompt you to upload the file
if not os.path.exists('dataset.jsonl'):
    print("Please upload your dataset.jsonl file:")
    uploaded = files.upload()

# 2. Load the JSONL dataset
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def format_chat_template(examples):
    texts = []
    for messages in examples["messages"]:
        # Apply Qwen's chat template
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_chat_template, batched=True)
print("\nSample training example:\n", dataset[0]['text'])

### 5. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # 3 epochs for 300 examples is good
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 6. Test Your New 'Macha' Model!

In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Bro explain what an API is"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])